# 02 — RotatE Embedding Baseline

Train RotatE on FB15k-237 with PyKEEN and evaluate with filtered ranking.

**No API key required.**

In [ ]:
import sys; sys.path.insert(0, '..')
import time
import numpy as np
import matplotlib.pyplot as plt
from src.data.fb15k237 import FB15k237Dataset
from src.models.embedding_baseline import RotatEBaseline
from src.eval.evaluate import evaluate
from src.eval.candidates import generate_candidates
from src.eval.metrics import compute_metrics
from src.utils.reproducibility import set_seed
from src.config import Settings

settings = Settings()
set_seed(settings.random_seed)
print('Settings loaded.')

## 1. Load Dataset

In [ ]:
dataset = FB15k237Dataset(data_dir=settings.data_dir)
dataset.load()
print(f'Train:{len(dataset.train):,}  Valid:{len(dataset.valid):,}  Test:{len(dataset.test):,}')

## 2. Train RotatE

In [ ]:
model = RotatEBaseline(
    embedding_dim=128, num_epochs=50,
    batch_size=512, lr=1e-3,
    random_seed=settings.random_seed,
    cache_dir=settings.cache_dir,
)
t0 = time.time()
model.fit(dataset)
print(f'Training done in {time.time()-t0:.1f}s')

## 3. Loss Curve

In [ ]:
if hasattr(model,'loss_history') and model.loss_history:
    plt.figure(figsize=(8,4))
    plt.plot(model.loss_history, color='steelblue')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.title('RotatE Training Loss')
    plt.tight_layout(); plt.show()
else:
    print('Loss history not exposed by this backend.')

## 4. Evaluate on Test Set (Filtered)

In [ ]:
results = evaluate(
    model=model, dataset=dataset, split='test',
    num_queries=settings.sample_test_queries, filtered=True,
)
print('=== Filtered Ranking Metrics ===')
for k,v in results.items(): print(f'  {k:<10}: {v:.4f}')

## 5. Top-K Candidates for a Sample Query

In [ ]:
h, r, t_true = dataset.test[0]
print(f'Query : ({h}, {r}, ?)')
print(f'Answer: {t_true}\n')
candidates = generate_candidates(
    model=model, head=h, relation=r,
    dataset=dataset, top_k=settings.num_candidates,
)
print(f'Top-{settings.topk_show} candidates:')
for i,(entity,score) in enumerate(candidates[:settings.topk_show],1):
    marker = '<-- ground truth' if entity==t_true else ''
    print(f'  {i:2}. {entity:<30} score={score:.4f}  {marker}')

## 6. MRR vs Top-K Budget

In [ ]:
ks=[5,10,25,50,100]; mrrs=[]
for k in ks:
    r=evaluate(model=model,dataset=dataset,split='test',num_queries=50,filtered=True,top_k=k)
    mrrs.append(r.get('mrr',0))
plt.figure(figsize=(7,4))
plt.plot(ks,mrrs,marker='o',color='steelblue')
plt.xlabel('Top-K'); plt.ylabel('MRR')
plt.title('Embedding MRR vs Candidate Budget')
plt.grid(True,alpha=0.3); plt.tight_layout(); plt.show()